In [37]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import json
import os
import sys
import logging
import traceback
import urllib.parse
from urllib.parse import urljoin, urlparse, quote
import urllib.request
from googlesearch import search

from permit_data_extraction.config import EXTERNAL_DATA_DIR

In [49]:
url = "https://permitsearch.epa.gov/oms-permit-hub/?media=air&type=TITLE_V"

In [50]:
response = requests.get(url)

In [51]:
response

<Response [200]>

In [52]:
soup = BeautifulSoup(response.content)

In [46]:
soup.find_all('a', href=True)

[]

In [54]:
print(soup.prettify())


<!DOCTYPE html>
<html data-beasties-container="" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Permit Hub | US EPA
  </title>
  <base href="/oms-permit-hub/"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <link href="favicon.ico" rel="icon" type="image/x-icon"/>
  <style>
   body{--esri-calcite-mode-name:"light"}@font-face{font-family:Avenir Next;src:url("./media/b8b15cdf-85d1-4120-8daa-48863d803939-2FPB52EI.woff2")format("woff2");font-weight:300;font-style:normal;font-display:auto}@font-face{font-family:Avenir Next;src:url("./media/09ab0626-bb45-4650-acc8-0182d693df02-LN7F76PF.woff2")format("woff2");font-weight:400;font-style:normal;font-display:auto}@font-face{font-family:Avenir Next;src:url("./media/b9c5b839-db56-4419-8fcb-6ab661babb1d-IOTVFYR4.woff2")format("woff2");font-weight:400;font-style:italic;font-display:auto}@font-face{font-family:Avenir Next;src:url("./media/12f4c786-0bef-4a48-b7c0-eebaa7591688-FGL2JOTR.woff2")format("woff2");font-

In [31]:
search_query = 'epa permit hub Sonoco Hickory, Inc. – Hickory Plant - 04691T33 - 2nd Step Modification'

In [35]:
search_results = []
# Get first 10 search results
for url in search(search_query, num_results=10, lang="en"):
    if "epa.gov/oms-permit-hub/permit" in url:
        search_results.append(url)

In [36]:
search_results

['https://permitsearch.epa.gov/oms-permit-hub/permit/9a6670b0-19bc-ef11-b8e8-001dd8001877']

In [38]:
csv_path = EXTERNAL_DATA_DIR / "permit-hub-report-2025-07-21.csv"

In [40]:
pd.read_csv(EXTERNAL_DATA_DIR / "permit-hub-report-2025-07-21.csv", encoding='latin-1')

,Permit Title,Permitting Authority,Region,Permit Action Title,Permit Type,NSR Permit Action,Title V Permit Action,Submission Category,Permit Number,Owner/Operator,...,Facility Type,Owner/Operator.1,Mobile No Fixed Address,Offshore,Permitting Authority Facility,Mailing Address 1,Mailing Address 2,Mailing City,Mailing State,Mailing Zip
0,"Sonoco Hickory, Inc.  Hickory Plant - 04691T3...",North Carolina DAQ,4,"Sonoco Hickory, Inc.  Hickory Plant - 04691T3...",Title V Operating Permit,NaN,Significant Modification,Final,04691T33,NaN,...,NaN,"Sonoco Hickory, Inc.",No,No,NaN,PO BOX 2029,NaN,HICKORY,NaN,28603
1,Thiele Kaolin Company - Deepstep Road Plant,Georgia EPD,4,Thiele Kaolin Company - Deepstep Road Plant,Title V Operating Permit,NaN,Administrative Amendment,Final,3295-303-0008-V-05-0,NaN,...,NaN,Thiele Kaolin Company,No,No,NaN,P.O. BOX 1056,NaN,SANDERSVILLE,NaN,31082
2,ID 007371 San Brn Cnty Solid Waste Mgmt - Mill...,South Coast AQMD,9,ID 007371 San Brn Cnty Solid Waste Mgmt - Mill...,Major NSR Permit (Nonattainment); Title V Oper...,Permit Revision,Significant Modification,Final,NaN,NaN,...,NaN,NaN,No,No,NaN,222 W. HOSPITALITY LANE,2ND FLOOR,SAN BERNARDINO,NaN,92415
3,"Sentinel Peak Resources, S1114, S1211974",San Joaquin Valley Unified APCD,9,"Sentinel Peak Resources, S1114, S1211974",Title V Operating Permit,NaN,Administrative Amendment,Final,S1114,S1211974,...,NaN,Jason Goklaney,No,No,NaN,NaN,NaN,NaN,NaN,NaN
4,TranscraftCorporation_TVRenewal_04172023,Kentucky DEP,4,TranscraftCorporation_TVRenewal_04172023,Title V Operating Permit,NaN,Renewal,Final,V-22-020,NaN,...,NaN,NaN,No,No,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9953,Griffin Lumber Company TV Renewal,Georgia EPD,4,Griffin Lumber Company TV Renewal,Title V Operating Permit,NaN,Renewal,Draft/Proposed for Concurrent Review,2421-081-0008-V-04-0,NaN,...,NaN,NaN,No,No,NaN,P.O. Box 237,NaN,Cordele,Georgia,31010
9954,Langdale Forest Products TV Renewal,Georgia EPD,4,Langdale Forest Products TV Renewal,Title V Operating Permit,NaN,Renewal,Draft/Proposed for Concurrent Review,2421-185-0009-V-05-0,NaN,...,NaN,NaN,No,No,NaN,P.O. BOX 1088,NaN,VALDOSTA,NaN,31603
9955,Eagle Point Landfill TV Renewal,Georgia EPD,4,Eagle Point Landfill TV Renewal,Minor NSR Permit (True Minor); Title V Operati...,Modification,Significant Modification,Draft/Proposed for Concurrent Review,4953-117-0059-V-04-2,NaN,...,NaN,NaN,No,No,NaN,NaN,NaN,NaN,NaN,NaN
9956,ID 800408 Northrop Grumman - de Minimis Signif...,South Coast AQMD,9,ID 800408 Northrop Grumman - de Minimis Signif...,Major NSR Permit (Nonattainment); Title V Oper...,Permit Revision,Significant Modification,Proposed,NaN,NaN,...,NaN,Northrop Grumman System Corporation,No,No,800408,One Space Park,Building S/1309,Redondo Beach,CA,90278
